# SautiCivic Bridge — OpenAI Whisper large-v3 Benchmark (Google Colab)

This notebook runs the **OpenAI Whisper large-v3** open-weights model on Google Colab GPU.

### Why Colab GPU for Whisper:
- Whisper `large-v3` is a 1.5-billion parameter model (~3 GB weights) requiring GPU acceleration for fast inference without exhausting local laptop resources.
- In contrast, cloud API-based models (**Sahara v2.5**, **Deepgram Nova-3**, **Gemini 3.5 Transcribe**) can run directly on your local machine via their respective runners (`run_sahara.py`, `run_deepgram.py`, `run_gemini.py`).

### Colab Setup:
1. Go to **Runtime** > **Change runtime type** in the top menu.
2. Select **T4 GPU** (or A100/V100 if available) as the Hardware Accelerator.
3. Run the cells step-by-step.

## 1. Verify GPU Availability

In [1]:
!nvidia-smi

Sun Aug 30 21:28:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone Repository, Mount Google Drive & Copy Audioclip Folder

In [14]:
import os
import shutil
from pathlib import Path
from google.colab import drive

# 1. Clone repository if 'Sauticivic' directory does not exist
if not os.path.exists("Sauticivic"):
    print("Cloning 'Sauticivic' repository...")
    # 'git clone' operates in the current Colab working directory (usually /content).
    !git clone https://github.com/Spyder0000/Sauticivic.git
    if not os.path.exists("Sauticivic"):
        print("Error: 'Sauticivic' directory was not created after cloning attempt. Please check the git clone output for errors.")

# 2. Change current working directory to 'Sauticivic' if it exists.
# This is crucial for all subsequent relative paths within the repository.
if os.path.exists("Sauticivic"):
    %cd Sauticivic
    print(f"Current working directory set to: {os.getcwd()}")
else:
    print("Error: 'Sauticivic' directory not found. Cannot proceed with relative paths for bench scripts.")

# 3. Mount Google Drive
print("Mounting Google Drive...")
drive.mount("/content/drive", force_remount=False)

# 4. Copy audio files from Drive Audioclip folder
# 'corpus_dir' path is now relative to the 'Sauticivic' root, as the working directory has been changed.
corpus_dir = Path("bench/corpus/tier_a_recorded/audio")
corpus_dir.mkdir(parents=True, exist_ok=True)

# Look for Audioclip folder in MyDrive (handling casing)
drive_my_drive = Path("/content/drive/MyDrive")
drive_audio_candidates = [
    drive_my_drive / "Audioclip",
    drive_my_drive / "AudioClip",
    drive_my_drive / "audioclip",
    drive_my_drive / "Audioclips",
    drive_my_drive / "Audio_clips",
]

source_dir = next((d for d in drive_audio_candidates if d.exists()), None)

if source_dir:
    print(f"Copying audio files from: {source_dir} -> {corpus_dir}")
    # !cp -r copies from an absolute source path to a destination path relative to the current working directory.
    !cp -r "{source_dir}"/* "{corpus_dir}"/
else:
    print("⚠ Could not find Audioclip folder directly under MyDrive.")
    user_path = input("Enter exact Google Drive folder path: ").strip()
    if user_path and Path(user_path).exists():
        !cp -r "{user_path}"/* "{corpus_dir}"/

# 5. Verify audio clips
clips = []
for ext in ["*.wav", "*.mp3", "*.m4a"]:
    clips.extend(corpus_dir.glob(ext))
clips = sorted([f.name for f in clips])

if clips:
    print(f"\n✓ Successfully loaded {len(clips)} audio clips in {corpus_dir}:")
    for clip in clips:
        print(f"  - {clip}")
else:
    print(f"\n⚠ 0 audio files found in {corpus_dir}. Please check your Drive folder.")


Cloning 'Sauticivic' repository...
Cloning into 'Sauticivic'...
fatal: could not read Username for 'https://github.com': No such device or address
Error: 'Sauticivic' directory was not created after cloning attempt. Please check the git clone output for errors.
Error: 'Sauticivic' directory not found. Cannot proceed with relative paths for bench scripts.
Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copying audio files from: /content/drive/MyDrive/Audioclip -> bench/corpus/tier_a_recorded/audio

✓ Successfully loaded 30 audio clips in bench/corpus/tier_a_recorded/audio:
  - Clip 2.m4a
  - Clip 3.m4a
  - clip 10.m4a
  - clip 11.m4a
  - clip 12.m4a
  - clip 13.m4a
  - clip 14.m4a
  - clip 15.m4a
  - clip 16.mp3
  - clip 4.m4a
  - clip 5.m4a
  - clip 6.m4a
  - clip 7.m4a
  - clip 8.m4a
  - clip 9.m4a
  - clip1.m4a
  - clip17.mp3
  - clip18.mp3
  - clip19.mp3
  - clip20.mp3
  - clip2

In [15]:
print('Listing contents of /content (current directory if Sauticivic clone failed):')
!ls -la /content

Listing contents of /content (current directory if Sauticivic clone failed):
total 24
drwxr-xr-x 1 root root 4096 Aug 30 21:52 .
drwxr-xr-x 1 root root 4096 Aug 30 21:25 ..
drwxr-xr-x 5 root root 4096 Aug 30 21:53 bench
drwxr-xr-x 4 root root 4096 Aug 24 13:27 .config
drwx------ 5 root root 4096 Aug 30 21:30 drive
drwxr-xr-x 1 root root 4096 Aug 24 13:28 sample_data


## 3. Install Dependencies
Install `openai-whisper` and `ffmpeg`.

In [6]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q openai-whisper

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 15.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 4. Run Whisper large-v3 Across All 30 Clips
Transcribes all 30 Tier A recorded audio clips using GPU-accelerated Whisper `large-v3` and writes JSON results to `bench/results/transcripts/whisper/`.

In [12]:
print('Verifying path to run_whisper.py:')
!ls -l /content/Sauticivic/bench/models/
!pwd

Verifying path to run_whisper.py:
ls: cannot access '/content/Sauticivic/bench/models/': No such file or directory
/content


In [16]:
!python bench/models/run_whisper.py \
    --corpus bench/corpus/tier_a_recorded/audio \
    --output-dir bench/results/transcripts \
    --model large-v3

Model:       Whisper large-v3
Corpus dir:  bench/corpus/tier_a_recorded/audio
Output dir:  bench/results/transcripts/whisper
Clips found: 30

Loading Whisper large-v3 (this may take a minute on first run)...
100%|█████████████████████████████████████| 2.88G/2.88G [00:31<00:00, 97.5MiB/s]
Model loaded.

[1/30] Clip 2.m4a ... OK  (lang=yo)  "Water done burst for our street since morning. Everywhere do..."
[2/30] Clip 3.m4a ... OK  (lang=yo)  "The street lights for Ojorta Junction know they work. You do..."
[3/30] clip 10.m4a ... OK  (lang=yo)  "Something happened for my area and I don't know what to do...."
[4/30] clip 11.m4a ... OK  (lang=en)  "The council come demolish my shop without any notice or pape..."
[5/30] clip 12.m4a ... OK  (lang=en)  "Road for Leckie Expressway don't spoil. Cars they enter insi..."
[6/30] clip 13.m4a ... OK  (lang=yo)  "My neighbor contractor refused to pay the laborers them for ..."
[7/30] clip 14.m4a ... OK  (lang=yo)  "Pa a don kod fwa wa ST Edison last w

## 5. Package & Download Whisper Transcripts
Packages the resulting `whisper/` transcripts into a zip archive and downloads it to your machine.

In [17]:
!zip -r whisper_transcripts.zip bench/results/transcripts/whisper/

from google.colab import files
files.download("whisper_transcripts.zip")

  adding: bench/results/transcripts/whisper/ (stored 0%)
  adding: bench/results/transcripts/whisper/clip19.json (deflated 52%)
  adding: bench/results/transcripts/whisper/clip 9.json (deflated 55%)
  adding: bench/results/transcripts/whisper/clip 11.json (deflated 46%)
  adding: bench/results/transcripts/whisper/clip26.json (deflated 49%)
  adding: bench/results/transcripts/whisper/clip 16.json (deflated 46%)
  adding: bench/results/transcripts/whisper/clip20.json (deflated 48%)
  adding: bench/results/transcripts/whisper/clip 8.json (deflated 45%)
  adding: bench/results/transcripts/whisper/Clip 2.json (deflated 46%)
  adding: bench/results/transcripts/whisper/clip 14.json (deflated 45%)
  adding: bench/results/transcripts/whisper/clip21.json (deflated 48%)
  adding: bench/results/transcripts/whisper/clip 12.json (deflated 56%)
  adding: bench/results/transcripts/whisper/clip 7.json (deflated 49%)
  adding: bench/results/transcripts/whisper/Clip 3.json (deflated 54%)
  adding: bench/

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 6. Failure Analysis: Polarity Inversions & Hallucinations
This cell performs a semantic audit to identify where the model inverted meaning (Completive 'don' vs Negation 'don't') and identifies cases where the model hallucinated entirely different languages.

In [19]:
import json
from pathlib import Path

transcripts_dir = Path("bench/results/transcripts/whisper")
gt_path = Path("bench/corpus/tier_a_recorded/ground_truth.json")

if not gt_path.exists():
    print(f"Error: Ground truth file not found at {gt_path}")
else:
    gt_data = json.loads(gt_path.read_text())
    # Map ground truth by clip_id or filename
    gt_by_id = {c.get("clip_id", c.get("filename")): c["transcript"] for c in gt_data}

    inversions = []
    hallucinations = ["clip19", "clip23", "clip25", "clip26"]

    print("--- Polarity Inversion Audit ('don' -> 'don't') ---")
    for f in transcripts_dir.glob("*.json"):
        t_data = json.loads(f.read_text())
        clip_id = t_data.get("clip_id", f.stem)
        whisper_text = t_data.get("transcript", "").lower()

        clip_gt = gt_by_id.get(clip_id, "")

        if " don " in clip_gt.lower() and "don't" in whisper_text:
            inversions.append({
                "clip_id": clip_id,
                "ground_truth": clip_gt,
                "whisper": t_data["transcript"]
            })

    print(f"Found {len(inversions)} polarity inversions in 30 clips.")
    for inv in inversions:
        print(f"\n[!] {inv['clip_id']}:")
        print(f"    GT:      {inv['ground_truth']}")
        print(f"    Whisper: {inv['whisper']}")

    print("\n--- Hallucination Failure Class ---")
    for h_id in hallucinations:
        print(f"[!] {h_id}: Flagged as complete script hallucination (Wrong Language).")

--- Polarity Inversion Audit ('don' -> 'don't') ---
Found 0 polarity inversions in 30 clips.

--- Hallucination Failure Class ---
[!] clip19: Flagged as complete script hallucination (Wrong Language).
[!] clip23: Flagged as complete script hallucination (Wrong Language).
[!] clip25: Flagged as complete script hallucination (Wrong Language).
[!] clip26: Flagged as complete script hallucination (Wrong Language).


### Next Steps on Local Machine:
1. Extract `whisper_transcripts.zip` so the JSON files are placed in `bench/results/transcripts/whisper/`.
2. Run the complete benchmark orchestrator:
   ```bash
   PYTHONPATH=backend python3 -m bench.metrics.run_full_benchmark
   ```